# Prueba manual de `oscrigol.py`
Notebook breve para verificar comunicación, configuración básica, adquisición y autoajuste vertical.


## 1. Imports
Carga la clase `oscrigol` desde este directorio o desde una carpeta `utils/` equivalente.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "oscrigol.py").exists():
    HERE = Path("oscilloscopes/rigol-MSO2102A").resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from oscrigol import oscrigol


## 2. Parámetros
Define conexión, trigger, escalas iniciales y modo de descarga.


In [ ]:
RUN_HARDWARE = False  # Cambiar a True para ejecutar comandos sobre el Rigol real.
IP_ADDRESS = "192.168.2.2"
USE_SOCKET = True
VISA_BACKEND = "pyvisa"  # "pyvisa" o "nivisa".

CHANNELS = (1,)
MDEPTH = 70000
ACQ = 1
TRIGGER_LEVEL = 100e-3
VSCALE = 50e-3
HSCALE = 10e-6


## 3. Crear y configurar osciloscopio
Instancia `oscrigol` con socket y deja listo `download_mode="fast"`.


In [ ]:
scope = oscrigol(
    ip_address=IP_ADDRESS,
    use_socket=USE_SOCKET,
    visa_backend=VISA_BACKEND,
)

scope.config(
    channels=CHANNELS,
    chanBand=("20M",),
    chanCoup=("AC",),
    chanInv=("OFF",),
    chanImp=("OMEG",),
    trigSource="CHAN1",
    trigCoup="AC",
    trigLevel=TRIGGER_LEVEL,
    trigSlope="POS",
    acquisition=ACQ,
    mdepth=MDEPTH,
    download_mode="fast",
    waveform_points_timeout=0.5,
    max_retries=2,
)


## 4. Verificar comunicación
Consulta `*IDN?` para confirmar que el instrumento responde.


In [ ]:
if RUN_HARDWARE:
    scope.initComm()
    try:
        print(scope.getID())
    finally:
        scope.closeComm()
else:
    print("RUN_HARDWARE=False; no se abre conexión real.")


## 5. Autoajuste vertical opcional
Prueba el autoajuste histórico (`measure`) o el nuevo cálculo desde RAW (`raw`).


In [ ]:
if RUN_HARDWARE:
    adjusted = scope.autoAdjustVertScale(
        channels=CHANNELS,
        mode="PEAK",
        n_iter=2,
        target_divisions=8.0,
        min_divisions=6.0,
        max_divisions=7.5,
        minmax_source="raw",  # usar "measure" para el método anterior.
        saturation_min_count=2,
        verbose=True,
    )
    print(adjusted)
else:
    print("RUN_HARDWARE=False; se omite autoajuste.")


## 6. Capturar una señal
Ejecuta la adquisición configurada y devuelve `t`, `v1` y `v2max`.


In [ ]:
if RUN_HARDWARE:
    t, v1, v2max = scope()
    print("samples:", np.asarray(v1).shape[-1], "v2max:", v2max)
else:
    t = np.linspace(-7 * HSCALE, 7 * HSCALE, MDEPTH)
    v1 = 0.05 * np.sin(2 * np.pi * 20e3 * t)
    v2max = np.nan
    print("Señal simulada para probar la celda de gráfico.")


## 7. Graficar señal
Visualiza la adquisición o la señal simulada si no hay hardware activo.


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(np.asarray(t) * 1e6, np.asarray(v1).reshape(-1) * 1e3)
plt.xlabel("tiempo [us]")
plt.ylabel("amplitud [mV]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.title("CH1")
